# Product Clustering — KMeans baseline

Fits the clustering model used by `src/clustering/cluster.py` / the API, inspects cluster contents, and produces `datasets/clusters_with_reviews.csv` — required by the summarization endpoint.

In [ ]:
import sys
sys.path.append('../..')

import pandas as pd
from src.clustering.cluster import fit_clusters

df = pd.read_csv('../../datasets/merged_reviews.csv')
print(f'{len(df)} reviews loaded')

In [ ]:
products = df[['product_id', 'product_title']].drop_duplicates(subset=['product_title']).reset_index(drop=True)

## Fit and persist the model
Saves to `models/B_clustering/kmeans_model.pkl` (used by the API's `assign_cluster()`). Set `n_clusters` based on the elbow plot from `01_EDA.ipynb` (default 5, within the 4-6 range).

In [ ]:
N_CLUSTERS = 5
products = products.dropna(subset=['product_title']).reset_index(drop=True)
cluster_ids, kmeans = fit_clusters(products['product_title'].astype(str).tolist(), n_clusters=N_CLUSTERS)
products['cluster_id'] = cluster_ids
products.to_csv('../../datasets/clusters.csv', index=False)
products['cluster_id'].value_counts().sort_index()

## Inspect cluster contents
Manually review these samples to assign a human-readable label per cluster.

In [ ]:
for cid in sorted(products['cluster_id'].unique()):
    print(f'\n--- Cluster {cid} ({(products.cluster_id == cid).sum()} products) ---')
    for t in products[products.cluster_id == cid]['product_title'].head(8):
        print('  -', t)

## Assign labels
After inspecting the printed samples above, edit this dict to reflect what each cluster actually contains, then it gets written to `models/B_clustering/cluster_labels.json` (used by the API).

In [ ]:
import json

cluster_labels = {
    '0': 'Ebook readers',
    '1': 'Batteries',
    '2': 'Accessories',
    '3': 'Non-electronics',
    '4': 'Other electronics',
    # add/remove keys if N_CLUSTERS != 5
}

with open('../../models/B_clustering/cluster_labels.json', 'w') as f:
    json.dump(cluster_labels, f, indent=2)
print('Saved cluster_labels.json')

## Build `clusters_with_reviews.csv`
Join review-level data with each product's cluster_id. This is the file `api/routers/summarization.py` reads to build recommendation articles per category.

In [ ]:
# Join on product_title (the trusted identifier), not product_id -
# see notebooks/00_data_merging_and_cleaning.ipynb for why product_id is unreliable.
reviews_with_clusters = df.merge(
    products[['product_title', 'cluster_id']], on='product_title', how='inner'
)
reviews_with_clusters.to_csv('../../datasets/clusters_with_reviews.csv', index=False)
print(f'Saved {len(reviews_with_clusters)} rows to datasets/clusters_with_reviews.csv')
reviews_with_clusters.head()

## Sanity check
Confirm every cluster has enough reviews to generate a meaningful article (top-3 + worst product).

In [ ]:
reviews_with_clusters.groupby('cluster_id').size().rename('review_count')